# try1 inference (0.83313 public, 0.8384 local)

Loads the saved XGBoost and LightGBM models and scores the raw test series. Nothing is trained here.

Both models read the same 977 base features (`pipeline/features_v4.py`), built straight from the raw series. The final score is the average of the two models' rank percentiles.

Output: `id,break_probability`

In [ ]:
#settings, change these paths
DATA = 'data'          #competition folder: test_series.parquet + sample_submission.csv
MODEL = 'try1/model'   #xgboost.ubj, lightgbm.txt, hyperparameters.json
OUT = 'try1_out'       #scratch folder, gets created
WORKERS = 2

In [ ]:
#imports
import os, sys, json, subprocess
import pandas as pd
import matplotlib.pyplot as plt
os.environ['CUDA_VISIBLE_DEVICES'] = ''   #cpu only

In [ ]:
#look at the new data
raw = pd.read_parquet(f'{DATA}/test_series.parquet')
print(raw.shape, raw['id'].nunique(), 'series')
raw.head()

In [ ]:
#plot a few, red line is the boundary we score
ids = raw['id'].unique()[:3]
fig, ax = plt.subplots(len(ids), 1, figsize=(10, 6))
for a, i in zip(ax, ids):
    s = raw[raw['id'] == i].sort_values('time')
    a.plot(s['value'].values, lw=.7)
    a.axvline((s['period'] == 0).sum(), ls='--', c='r')
    a.set_title(str(i))
plt.tight_layout()

In [ ]:
#the saved models
import xgboost as xgb
booster = xgb.Booster(model_file=f'{MODEL}/xgboost.ubj')
print(len(booster.feature_names), 'features')
print(json.dumps(json.load(open(f'{MODEL}/hyperparameters.json')), indent=1))
print('5-fold auc', json.load(open(f'{MODEL}/submission_info.json'))['cv_auc'])

In [ ]:
#run it:
#1 raw series -> 977 base features
#2 xgboost and lightgbm predict
#3 rank average, write id,break_probability
subprocess.run([sys.executable, 'try1/v4.py', 'predict', '--data', DATA, '--out-dir', OUT,
                '--workers', str(WORKERS)], check=True)

In [ ]:
#check the output
sub = pd.read_csv(f'{OUT}/submission.csv', dtype={'id': str})
print(sub.shape)
print(sub.break_probability.describe())
sub.head()

In [ ]:
#compare with the csv that was submitted to kaggle
ref = pd.read_csv(f'{MODEL}/submission.csv', dtype={'id': str}).set_index('id').loc[sub.id, 'break_probability']
print('max difference', (ref.to_numpy() - sub.break_probability.to_numpy()).__abs__().max())